# Module 04: Data Cleaning & Preprocessing
**Synthetic payments case study**. Run cells in order. Outputs are written to `outputs/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('pandas', pd.__version__)
paths = [Path('data/messy_transactions.csv'), Path('messy_transactions.csv'), Path('04_Data_Cleaning/data/messy_transactions.csv')]
source = next((x for x in paths if x.exists()), None)
if source is None:
    raise FileNotFoundError('Upload messy_transactions.csv to Colab or run notebook in module folder.')
raw = pd.read_csv(source, dtype=str)
print('Raw shape:', raw.shape)
display(raw.head())

## 1. Profile before cleaning
Inspect missingness, raw categories and duplicate rows.

In [ ]:
profile = pd.DataFrame({'missing_count': raw.isna().sum(), 'missing_pct': (raw.isna().mean()*100).round(1), 'unique_values': raw.nunique(dropna=False)})
display(profile)
print('Exact duplicate rows:', int(raw.duplicated().sum()))
print('Status values:')
print(raw['payment_status'].value_counts(dropna=False))

## 2. Preserve raw values and normalize fields
Remove exact duplicates first. Ambiguous identifier collisions are handled separately.

In [ ]:
df = raw.drop_duplicates().copy()
exact_removed = len(raw) - len(df)
df['amount'] = pd.to_numeric(df['amount_usd'].str.replace(',', '', regex=False), errors='coerce')
df['status_clean'] = df['payment_status'].str.strip().str.casefold().replace({'complete':'completed'})
df['category_clean'] = df['category'].str.strip().str.casefold().replace({'grocery':'groceries'}).str.title()
df['city_clean'] = df['city'].str.strip().str.title()
iso = pd.to_datetime(df['transaction_date'], format='%Y-%m-%d', errors='coerce')
slash = pd.to_datetime(df['transaction_date'], format='%Y/%m/%d', errors='coerce')
df['date_clean'] = iso.fillna(slash)
print('Exact duplicates removed:', exact_removed)
display(df[['amount_usd','amount','payment_status','status_clean','transaction_date','date_clean']].head(15))

## 3. Validate rows with explicit reason codes
This example treats purchases as positive amounts. High amounts are flagged for review, not rejected. Conflicting transaction IDs are quarantined.

In [ ]:
conflicting_ids = set(df.loc[df.duplicated('transaction_id', keep=False), 'transaction_id'].dropna())
def reasons(row):
    problems = []
    if pd.isna(row['transaction_id']) or not str(row['transaction_id']).strip(): problems.append('missing_transaction_id')
    if pd.isna(row['customer_id']) or not str(row['customer_id']).strip(): problems.append('missing_customer_id')
    if row['transaction_id'] in conflicting_ids: problems.append('conflicting_transaction_id')
    if pd.isna(row['date_clean']): problems.append('invalid_or_missing_date')
    if pd.isna(row['amount']): problems.append('invalid_or_missing_amount')
    elif row['amount'] <= 0: problems.append('nonpositive_purchase_amount')
    if row['status_clean'] not in {'completed','pending','failed'}: problems.append('unknown_status')
    if pd.isna(row['category_clean']) or not str(row['category_clean']).strip(): problems.append('missing_category')
    return problems

df['reasons'] = df.apply(reasons, axis=1)
df['review_high_amount'] = df['amount'].gt(10000)
quarantine = df[df['reasons'].str.len().gt(0)].copy()
clean = df[df['reasons'].str.len().eq(0)].copy()
quarantine['reason_codes'] = quarantine['reasons'].apply(lambda x: ';'.join(x))
print({'raw':len(raw),'exact_removed':exact_removed,'clean':len(clean),'quarantined':len(quarantine)})
assert len(raw) == exact_removed + len(clean) + len(quarantine)
assert clean['transaction_id'].is_unique
assert clean['amount'].gt(0).all()
display(quarantine[['transaction_id','reason_codes']])

## 4. Feature engineering and business summaries
Use only information available at the intended prediction time.

In [ ]:
clean['transaction_month'] = clean['date_clean'].dt.to_period('M').astype(str)
clean['transaction_weekday'] = clean['date_clean'].dt.day_name()
completed = clean[clean['status_clean'].eq('completed')]
summary = completed.groupby('category_clean', as_index=False).agg(transaction_count=('transaction_id','size'),total_amount=('amount','sum')).sort_values('total_amount',ascending=False)
display(summary)
ax = summary.plot.bar(x='category_clean', y='total_amount', legend=False, title='Synthetic completed purchase totals')
ax.set_ylabel('USD')
plt.tight_layout(); plt.show()

## 5. Export clean data, quarantine and a quality report

In [ ]:
out = Path('outputs');out.mkdir(exist_ok=True)
clean_export = clean.drop(columns=['reasons']).copy()
clean_export.to_csv(out/'clean_transactions.csv', index=False)
quarantine.drop(columns=['reasons']).to_csv(out/'quarantined_transactions.csv', index=False)
summary.to_csv(out/'category_summary.csv', index=False)
report = pd.DataFrame([{'raw_rows':len(raw),'exact_duplicates_removed':exact_removed,'accepted_rows':len(clean),'quarantined_rows':len(quarantine),'high_amount_flags_in_accepted':int(clean['review_high_amount'].sum())}])
report.to_csv(out/'data_quality_report.csv',index=False)
display(report)
print('Saved files:', [str(x) for x in out.glob('*.csv')])

## 6. Train/test-safe preprocessing demonstration
Fit the imputer and scaler on training data only. This is a preprocessing demonstration, **not** a trained predictive model.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
X = clean[['amount']]
train, test = train_test_split(X, test_size=0.25, random_state=42)
pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler',StandardScaler())])
train_scaled = pipe.fit_transform(train)
test_scaled = pipe.transform(test)
print('Train/test rows:',len(train_scaled),len(test_scaled))
print('Train mean after scaling:',round(float(train_scaled.mean()),6))

## 7. Further work
Try the 15 exercises. In production, add schema checks, monitoring, tests, and business-approved rules. Synthetic records are not evidence of real fraud.